In [ ]:
import vowpalwabbit 

vw = vowpalwabbit.Workspace("--cb_explore 4 --epsilon 0.2 --quiet")

In [1]:
import numpy as np
import vowpalwabbit


class OnlineContextualBandit:
    """
    Basic online contextual bandit using Vowpal Wabbit.

    Context:
        fixed-size numeric vector

    Actions:
        fixed set {1, 2, ..., num_actions}

    Interaction:
        context -> action
        environment -> reward
        reward -> learner update
    """

    def __init__(
        self,
        context_dim: int,
        num_actions: int,
        epsilon: float = 0.1,
        seed: int = 42,
    ):
        if context_dim <= 0:
            raise ValueError("context_dim must be > 0")

        if num_actions <= 1:
            raise ValueError("num_actions must be > 1")

        if not 0.0 <= epsilon <= 1.0:
            raise ValueError("epsilon must be between 0 and 1")

        self.context_dim = context_dim
        self.num_actions = num_actions

        # VW learner
        self.vw = vowpalwabbit.Workspace(
            f"--cb_explore {num_actions} "
            f"--epsilon {epsilon} "
            "--quiet"
        )

        # We perform the actual sampling ourselves from VW's PMF.
        self.rng = np.random.default_rng(seed)

    def _check_context(self, context):
        """
        VW itself does not enforce a fixed vector dimension.
        We enforce it at our application boundary.
        """

        context = np.asarray(context, dtype=np.float32)

        if context.ndim != 1:
            raise ValueError(
                f"context must be 1-D, got shape {context.shape}"
            )

        if len(context) != self.context_dim:
            raise ValueError(
                f"expected context dimension {self.context_dim}, "
                f"got {len(context)}"
            )

        return context

    def _context_to_vw(self, context):
        """
        Convert

            [0.2, 0.8]

        into

            | x0:0.2 x1:0.8
        """

        context = self._check_context(context)

        features = " ".join(
            f"x{i}:{value}"
            for i, value in enumerate(context)
        )

        return f"| {features}"

    def predict(self, context):
        """
        Given a context:

            1. ask VW for an action PMF
            2. sample one action from that PMF
            3. return the action and its selection probability
        """

        vw_example = self._context_to_vw(context)

        # VW returns the probability mass function
        pmf = self.vw.predict(vw_example)

        pmf = np.asarray(pmf, dtype=np.float64)

        if len(pmf) != self.num_actions:
            raise RuntimeError(
                f"VW returned {len(pmf)} probabilities, "
                f"expected {self.num_actions}"
            )

        # Numerical safety
        total = pmf.sum()

        if total <= 0:
            raise RuntimeError("VW returned an invalid probability distribution")

        pmf /= total

        # Sample according to VW's policy distribution.
        action_index = self.rng.choice(
            self.num_actions,
            p=pmf,
        )

        # VW's fixed-action CB labels use action IDs 1..K.
        action = action_index + 1

        # Probability with which THIS particular action was selected.
        probability = float(pmf[action_index])

        return action, probability

    def learn(self, context, action, reward, probability):
        """
        Send observed feedback back to VW.

        VW expects:

            action:cost:probability | features

        Since VW minimizes cost:

            cost = -reward
        """

        if not 1 <= action <= self.num_actions:
            raise ValueError(
                f"action must be in [1, {self.num_actions}]"
            )

        if probability <= 0:
            raise ValueError("probability must be > 0")

        cost = -float(reward)

        vw_example = (
            f"{action}:{cost}:{probability} "
            f"{self._context_to_vw(context)}"
        )

        # THIS is the online learning update.
        self.vw.learn(vw_example)

    def close(self):
        self.vw.finish()

In [2]:
import numpy as np


rng = np.random.default_rng(123)


def environment(context, action):
    """
    Hidden environment.

    The bandit does NOT know these reward equations.
    """

    x0, x1 = context

    if action == 1:
        expected_reward = 0.9 - 0.8 * x1

    elif action == 2:
        expected_reward = 0.1 + 0.8 * x1

    elif action == 3:
        expected_reward = 0.45 + 0.3 * x0

    else:
        raise ValueError("invalid action")

    # Bernoulli reward
    reward = float(rng.random() < expected_reward)

    return reward

In [4]:
agent = OnlineContextualBandit(
    context_dim=2,
    num_actions=3,
    epsilon=0.1,
    seed=42,
)

rng_context = np.random.default_rng(999)

num_steps = 10_000

total_reward = 0.0

try:
    for step in range(1, num_steps + 1):

        # --------------------------------
        # 1. Environment gives context
        # --------------------------------

        context = rng_context.random(2)

        # --------------------------------
        # 2. Agent chooses action
        # --------------------------------

        action, probability = agent.predict(context)

        # --------------------------------
        # 3. Environment returns reward
        # --------------------------------

        reward = environment(context, action)

        # --------------------------------
        # 4. Give reward back to VW
        # --------------------------------

        agent.learn(
            context=context,
            action=action,
            reward=reward,
            probability=probability,
        )

        total_reward += reward

        if step % 1000 == 0:
            avg_reward = total_reward / step

            print(
                f"step={step:5d} "
                f"average_reward={avg_reward:.4f}"
            )

finally:
    agent.close()

step= 1000 average_reward=0.6210
step= 2000 average_reward=0.6485
step= 3000 average_reward=0.6633
step= 4000 average_reward=0.6690
step= 5000 average_reward=0.6778
step= 6000 average_reward=0.6817
step= 7000 average_reward=0.6847
step= 8000 average_reward=0.6835
step= 9000 average_reward=0.6802
step=10000 average_reward=0.6793
